# Getting rotation between LiDAR and base_link based in coupled IMU orientation in a flat garage

In [ ]:
from scipy.spatial.transform import Rotation

From `vilma_tools/scripts/average_orientation.py` processing `rosbag2_2026_05_26-14_57_58`:

```shell
========== RESULTS ==========
Samples: 4204
Average quaternion [x y z w]: [-0.05453805 -0.02045581  0.96923066 -0.23916332]
Average roll  (deg): -0.782524
Average pitch (deg): 6.632722
Average yaw   (deg): -152.323010
=============================
```

IMU-LiDAR calibration:

```yaml
imu_link:
    x: 0.006
    y: -0.011
    z: 0.065
    roll: 0.0017453
    pitch: -0.0001745
    yaw: 0.0684169
```

Where:

```xml
 <!-- imu -->
    <xacro:imu_macro
      name="imu"
      parent="sensor_kit_base_link"
      namespace=""
      x="${calibration['sensor_kit_base_link']['imu_link']['x']}"
      y="${calibration['sensor_kit_base_link']['imu_link']['y']}"
      z="${calibration['sensor_kit_base_link']['imu_link']['z']}"
      roll="${calibration['sensor_kit_base_link']['imu_link']['roll']}"
      pitch="${calibration['sensor_kit_base_link']['imu_link']['pitch']}"
      yaw="${calibration['sensor_kit_base_link']['imu_link']['yaw']}"
      fps="100"
    />
```


In [ ]:
q_imu_earth = [-0.05453805, -0.02045581,  0.96923066, -0.23916332]

# IMU measurement in earth reference
r_imu_earth = Rotation.from_quat(q_imu_earth)

print('r_imu_earth')
print(r_imu_earth.as_euler('xyz'))

_, _, yaw_imu_earth = r_imu_earth.as_euler('xyz')

# Converting to local plane
# world -> imu = yaw * tilt [euler xyz = RzRyRx]
# tilt = yaw^-1 * (world -> imu)
r_imu_earth_local = Rotation.from_euler('z', yaw_imu_earth).inv() *  r_imu_earth

print('r_imu_earth_local')
print(r_imu_earth_local.as_euler('xyz'))

# Rotation of IMU from LiDAR frame
r_imu_lidar = Rotation.from_euler(
    'xyz',
    [0.0017453, -0.0001745, 0.0684169]
)

print('r_imu_lidar')
print(r_imu_lidar.as_euler('xyz'))

# I must remove the rotation from IMU to LiDAR to get the rotation from the LiDAR to local plane
r_lidar_earth_local = r_imu_earth_local *  r_imu_lidar.inv()

print('r_lidar_earth_local')
print(r_lidar_earth_local.as_euler('xyz'))

_, _, yaw_lidar_earth_local = r_lidar_earth_local.as_euler('xyz')

# Removing yaw because it is IMU-LiDAR yaw, not LiDAR-local_frame (base_link)
r_lidar_earth_local_  = Rotation.from_euler('z', yaw_lidar_earth_local).inv() * r_lidar_earth_local 


print('r_lidar_earth_local_')
print(r_lidar_earth_local_.as_euler('xyz'))



r_imu_earth
[-0.01365763  0.11576284 -2.65853806]
r_imu_earth_local
[-1.36576269e-02  1.15762842e-01 -8.28454326e-17]
r_imu_lidar
[ 0.0017453 -0.0001745  0.0684169]
r_lidar_earth_local
[-0.02332672  0.11461199 -0.06886397]
r_lidar_earth_local_
[-2.33267209e-02  1.14611985e-01 -5.02026679e-18]


Then:

```yaml
sensor_kit_base_link:
    roll: -2.33267209e-02
    pitch: 1.14611985e-01
```